In [ ]:
from pathlib import Path

import numpy as np
import pymeshlab
import tifffile

In [ ]:
base_path = Path(r"D:\Tracking\NucleiTracking\data\interim\lightsheet\2025_02_06\recon")
arr = tifffile.imread(base_path / "recon_fused_tp_368_ch_0.tif")

In [ ]:
from skimage.feature import peak_local_max
from skimage.filters import difference_of_gaussians

dog = difference_of_gaussians(arr, 2, 6)
peaks = peak_local_max(dog, min_distance=5, threshold_abs=35)

In [ ]:
import napari
#
# viewer = napari.Viewer()
#
# points = viewer.add_points(peaks[:], size=5)
# img = viewer.add_image(arr, name="Image")
#
# # napari.run()

In [ ]:
import os

import igl
import matplotlib as mpl
import matplotlib.pyplot as plt
import pymeshlab
from blender_tissue_cartography import interface_pymeshlab as intmsl
from blender_tissue_cartography import interpolation as tcinterp
from blender_tissue_cartography import io as tcio
from blender_tissue_cartography import mesh as tcmesh
from blender_tissue_cartography import morphsnakes
from blender_tissue_cartography import registration as tcreg
from blender_tissue_cartography import remesh as tcremesh
from blender_tissue_cartography import remesh_pymeshlab as tcremesh_pymeshlab
from scipy import ndimage
from skimage import transform

In [ ]:
from scipy.stats import mode
from sklearn.cluster import DBSCAN

dbscan = DBSCAN(eps=15, min_samples=1)
vals = dbscan.fit_predict(peaks)
vmax = mode(vals)
vmax = vmax.mode
print(vmax)

In [ ]:
# viewer = napari.Viewer()

# colors = ["black" if val == vmax else "red" for val in vals]
# viewer.add_points(peaks[:], face_color=colors, size=5)
# # napari.run()

In [ ]:
point_cloud = tcmesh.ObjMesh(vertices=peaks[vals == vmax], faces=[])
point_cloud_pymeshlab = intmsl.convert_to_pymeshlab(point_cloud)

In [ ]:
ms = pymeshlab.MeshSet()
ms.add_mesh(point_cloud_pymeshlab)

ms.compute_normal_for_point_clouds(k=20, smoothiter=2)
ms.generate_surface_reconstruction_screened_poisson(
    depth=8,
    fulldepth=5,
)

ms.meshing_isotropic_explicit_remeshing(
    iterations=10, targetlen=pymeshlab.PercentageValue(1)
)

mesh_reconstructed = intmsl.convert_from_pymeshlab(ms.current_mesh())

In [ ]:
mesh_reconstructed.faces.shape

In [ ]:
mesh_reconstructed.write_obj(base_path / "dog_peaks.obj")

In [ ]:
mesh_reconstructed = tcremesh_pymeshlab.reconstruct_poisson(
    peaks, samplenum=1000, reconstruc_args={"depth": 8, "fulldepth": 5}
)
mesh_reconstructed.faces.shape

In [ ]:
mesh_reconstructed.write_obj("test_dog_peaks2.obj")

In [ ]:
mesh_uv = tcmesh.ObjMesh.read_obj(str(base_path.parent / "bottom.obj"))
# mesh_uv = tcmesh.ObjMesh.read_obj(r"D:\Tracking\NucleiTracking\data\interim\lightsheet\2025_02_06\small.obj")
normal_offsets = np.linspace(-12, 8, 21)
mapping_arr = tifffile.imread(base_path / "recon_fused_tp_151_ch_0.tif")

In [ ]:
# projected_data, projected_coordinates, projected_normals = tcinterp.create_cartographic_projections(
#     image=np.expand_dims(mapping_arr, axis=0),
#     mesh=mesh_uv,
#     resolution=(1, 1, 1),
#     normal_offsets=normal_offsets,
#     uv_grid_steps=1024)
# print("Image shape:", projected_data.shape)

mesh = mesh_uv

print(mesh.texture_vertices)

print(igl.flipped_triangles(mesh.texture_vertices, mesh.texture_tris))
image = np.expand_dims(mapping_arr, axis=0)
uv_grid_steps = 1024
map_back = True
use_fallback = "auto"
resolution = (1, 1, 1)

interpolated_3d_positions = tcinterp.interpolate_per_vertex_field_to_UV(
    mesh,
    mesh.vertices,
    domain="per-vertex",
    uv_grid_steps=uv_grid_steps,
    distance_threshold=0.0000001,
    map_back=map_back,
    use_fallback=use_fallback,
)
interpolated_normals = tcinterp.interpolate_per_vertex_field_to_UV(
    mesh,
    mesh.normals,
    domain="per-vertex",
    uv_grid_steps=uv_grid_steps,
    distance_threshold=0.0000001,
    map_back=map_back,
    use_fallback=use_fallback,
)
projected_data = tcinterp.interpolate_volumetric_data_to_uv_multilayer(
    image, interpolated_3d_positions, interpolated_normals, normal_offsets, resolution
)
print("Image shape:", projected_data.shape)

In [ ]:
full_locs = np.expand_dims(interpolated_3d_positions, 0) + np.expand_dims(
    interpolated_normals, 0
) * np.expand_dims(np.array(normal_offsets), [1, 2])

In [ ]:
c
# plt.imshow(maxp, cmap="gray")
# plt.show()
tifffile.imwrite(base_path / "long2b.tif", maxp, imagej=True)

In [ ]:
from skimage.feature import peak_local_max
from skimage.filters import difference_of_gaussians

dog = difference_of_gaussians(maxp, 4, 10)
peaks = peak_local_max(dog, min_distance=5, threshold_abs=15)

plt.imshow(maxp, cmap="gray")
plt.scatter(peaks[:, 1], peaks[:, 0], c="red", s=1)